In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


In [3]:
val_binary = pd.read_csv("Extraction/val_binary.tsv", sep="\t")
test_binary = pd.read_csv("Extraction/test_binary.tsv", sep="\t")
metadata_cols = ["Entry", "Length", "Sequence"]
concept_cols = [c for c in val_binary.columns if c not in metadata_cols]

In [4]:
def calculate_f1_array(precision, recall):
    denom = precision + recall
    return np.divide(
        2 * precision * recall,
        denom,
        out=np.zeros_like(denom, dtype=float),
        where=denom > 0
    )

def compare_features_to_concepts_fast(
    A,
    binary_df,
    concept_cols,
    thresholds=(0, 0.15, 0.5, 0.6, 0.8),
):
    """
    A: normalized activations, shape [n_proteins, n_features]
    binary_df: val_binary/test_binary
    concept_cols: concept label columns

    Returns dataframe with:
    concept, feature, threshold, precision, recall, f1
    """

    A = np.asarray(A)
    Y = binary_df[concept_cols].values.astype(bool)

    n_proteins, n_features = A.shape
    n_concepts = Y.shape[1]

    positives = Y.sum(axis=0)  # [n_concepts]
    results = []

    for threshold in thresholds:
        A_bin = A > threshold  # [n_proteins, n_features]

        # Matrix multiplication gives TP:
        # Y.T: [n_concepts, n_proteins]
        # A_bin: [n_proteins, n_features]
        # tp: [n_concepts, n_features]
        tp = Y.T.astype(np.int32) @ A_bin.astype(np.int32)

        pred_pos = A_bin.sum(axis=0)  # [n_features]
        fp = pred_pos[None, :] - tp

        precision = np.divide(
            tp,
            tp + fp,
            out=np.zeros_like(tp, dtype=float),
            where=(tp + fp) > 0,
        )

        recall = np.divide(
            tp,
            positives[:, None],
            out=np.zeros_like(tp, dtype=float),
            where=positives[:, None] > 0,
        )

        f1 = calculate_f1_array(precision, recall)

        concept_idx, feature_idx = np.nonzero(tp > 0)

        df_t = pd.DataFrame({
            "concept": [concept_cols[i] for i in concept_idx],
            "feature": feature_idx,
            "threshold": threshold,
            "precision": precision[concept_idx, feature_idx],
            "recall": recall[concept_idx, feature_idx],
            "f1": f1[concept_idx, feature_idx],
            #"tp": tp[concept_idx, feature_idx],
            #"fp": fp[concept_idx, feature_idx],
            #"positive_labels": positives[concept_idx],
        })

        results.append(df_t)

    return pd.concat(results, ignore_index=True)

CLS 8

In [8]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_8/embeddings_cls_nmf_8_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_8/embeddings_cls_nmf_8_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 8)
(25000, 8)
val f1: 0.06964
test f1: 0.06626
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,7,0.15,0.34213,0.953378,0.503554,0.34381,0.954601,0.505543


CLS 32

In [10]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_32/embeddings_cls_nmf_32_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_32/embeddings_cls_nmf_32_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 32)
(25000, 32)
val f1: 0.09601
test f1: 0.08921
Validation pairs with F1 > 0.5: 0
Those also with test F1 > 0.5: 0
Survival rate: 0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


CLS 128

In [11]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_128/embeddings_cls_nmf_128_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_128/embeddings_cls_nmf_128_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 128)
(25000, 128)
val f1: 0.1183
test f1: 0.10661
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,transmembrane,23,0.15,0.449396,0.658193,0.534114,0.444814,0.641755,0.525437
1,GO:0005634,55,0.15,0.383387,0.796113,0.517540,0.384464,0.795161,0.518318


In [33]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_320/embeddings_cls_nmf_320_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_320/embeddings_cls_nmf_320_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 320)
(25000, 320)
val f1: 0.1318
test f1: 0.1176
Validation pairs with F1 > 0.5: 3
Those also with test F1 > 0.5: 1
Survival rate: 0.3333333333333333


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003925,40,0.5,0.533784,0.721461,0.613592,0.495868,0.659341,0.566038


Layer Mean

In [12]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_8/embeddings_layer_mean_nmf_8_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_8/embeddings_layer_mean_nmf_8_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 8)
(25000, 8)
val f1: 0.0832
test f1: 0.0805
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,transmembrane,0,0.15,0.666191,0.721230,0.692619,0.668414,0.721118,0.693766
1,GO:0005634,3,0.50,0.433307,0.676906,0.528382,0.438747,0.687373,0.535614


In [13]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_16/embeddings_layer_mean_nmf_16_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_16/embeddings_layer_mean_nmf_16_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 16)
(25000, 16)
val f1: 0.11202
test f1: 0.10633
Validation pairs with F1 > 0.5: 4
Those also with test F1 > 0.5: 3
Survival rate: 0.75


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,transmembrane,2,0.15,0.930411,0.521299,0.668208,0.923006,0.513713,0.660059
1,GO:0005634,3,0.15,0.412905,0.758461,0.534713,0.408441,0.745820,0.527825
3,GO:0007186,10,0.50,0.531250,0.488038,0.508728,0.548387,0.483412,0.513854


In [14]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_32/embeddings_layer_mean_nmf_32_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_32/embeddings_layer_mean_nmf_32_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 32)
(25000, 32)
val f1: 0.13109
test f1: 0.12546
Validation pairs with F1 > 0.5: 9
Those also with test F1 > 0.5: 6
Survival rate: 0.6666666666666666


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005506,15,0.80,1.000000,0.503650,0.669903,1.000000,0.491935,0.659459
1,heme,15,0.80,0.985507,0.485714,0.650718,0.967213,0.460938,0.624339
2,1.14,15,0.80,0.821256,0.419753,0.555556,0.846995,0.411141,0.553571
4,GO:0005634,20,0.15,0.417961,0.742694,0.534900,0.418118,0.730869,0.531929
3,GO:0020037,15,0.80,1.000000,0.380515,0.551265,1.000000,0.353282,0.522111
7,GO:0007186,18,0.50,0.540761,0.476077,0.506361,0.548747,0.466825,0.504481


In [15]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_64/embeddings_layer_mean_nmf_64_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_64/embeddings_layer_mean_nmf_64_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 64)
(25000, 64)
val f1: 0.15431
test f1: 0.1446
Validation pairs with F1 > 0.5: 15
Those also with test F1 > 0.5: 12
Survival rate: 0.8


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,pyridoxal 5'-phosphate,46,0.60,0.918239,0.640351,0.754522,0.954248,0.616034,0.748718
1,GO:0005506,18,0.50,1.000000,0.508516,0.674194,0.994652,0.500000,0.665474
2,heme,18,0.50,0.985646,0.490476,0.655008,0.962567,0.468750,0.630473
5,abc transporter,11,0.50,1.000000,0.446429,0.617284,0.947368,0.444444,0.605042
4,ig-like,22,0.50,0.612108,0.643868,0.627586,0.600962,0.608273,0.604595
3,GO:0003925,32,0.50,0.551370,0.735160,0.630137,0.511905,0.708791,0.594470
6,fad,35,0.50,0.761905,0.450704,0.566372,0.772947,0.448179,0.567376
7,transmembrane,1,0.15,0.956021,0.399519,0.563537,0.944928,0.391155,0.553279
11,GO:0005634,47,0.15,0.419079,0.806715,0.551606,0.421098,0.805220,0.552999
9,1.14,18,0.50,0.822967,0.424691,0.560261,0.828877,0.411141,0.549645


In [16]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_128/embeddings_layer_mean_nmf_128_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_128/embeddings_layer_mean_nmf_128_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 128)
(25000, 128)
val f1: 0.18572
test f1: 0.17325
Validation pairs with F1 > 0.5: 26
Those also with test F1 > 0.5: 23
Survival rate: 0.8846153846153846


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,6.2,63,0.80,0.941176,0.820513,0.876712,0.912281,0.764706,0.832000
1,pyridoxal 5'-phosphate,79,0.50,0.993333,0.653509,0.788360,1.000000,0.662447,0.796954
2,GO:0003925,28,0.60,0.684211,0.890411,0.773810,0.601562,0.846154,0.703196
3,n-acetyltransferase,114,0.50,0.895833,0.661538,0.761062,0.941176,0.551724,0.695652
4,ig-like,92,0.50,0.954887,0.599057,0.736232,0.919028,0.552311,0.689970
13,abc transporter,64,0.15,0.417910,1.000000,0.589474,0.515924,1.000000,0.680672
5,GO:0003924,28,0.50,0.920000,0.587591,0.717149,0.920266,0.528626,0.671515
7,protein kinase,70,0.50,0.639889,0.686139,0.662207,0.653008,0.684018,0.668153
6,GO:0005506,72,0.50,1.000000,0.506083,0.672052,1.000000,0.500000,0.666667
8,heme,72,0.60,0.985577,0.488095,0.652866,0.967391,0.463542,0.626761


In [34]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_320/embeddings_layer_mean_nmf_320_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_320/embeddings_layer_mean_nmf_320_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 320)
(25000, 320)
val f1: 0.20229
test f1: 0.19187
Validation pairs with F1 > 0.5: 32
Those also with test F1 > 0.5: 31
Survival rate: 0.96875


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
3,c-type lectin,173,0.50,0.968254,0.709302,0.818792,1.000000,0.759036,0.863014
1,protein kinase,121,0.50,0.990850,0.750495,0.854085,0.985882,0.765297,0.861697
2,cadherin,219,0.80,1.000000,0.728814,0.843137,1.000000,0.692308,0.818182
5,pyridoxal 5'-phosphate,212,0.50,0.993289,0.649123,0.785146,1.000000,0.658228,0.793893
0,6.2,52,0.60,0.902778,0.833333,0.866667,0.791045,0.779412,0.785185
7,GO:0106310,121,0.50,0.779085,0.741294,0.759720,0.760000,0.760895,0.760447
4,n-acetyltransferase,146,0.50,0.940000,0.723077,0.817391,0.972222,0.603448,0.744681
9,GO:0004674,121,0.50,0.671895,0.689933,0.680795,0.681176,0.721945,0.700969
6,GO:0003925,28,0.60,0.680702,0.885845,0.769841,0.603175,0.835165,0.700461
20,abc transporter,179,0.15,0.421053,1.000000,0.592593,0.503106,1.000000,0.669421


MAX

In [18]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_8/embeddings_max_nmf_8_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_max_8/embeddings_max_nmf_8_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 8)
(25000, 8)
val f1: 0.08491
test f1: 0.08121
Validation pairs with F1 > 0.5: 4
Those also with test F1 > 0.5: 4
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003924,5,0.8,0.847561,0.507299,0.634703,0.864865,0.488550,0.624390
1,protein kinase,0,0.8,0.463899,0.763366,0.577096,0.485477,0.747945,0.588785
2,GO:0005525,5,0.8,0.945122,0.412234,0.574074,0.956081,0.403709,0.567703
3,GO:0106310,0,0.8,0.385680,0.797264,0.519870,0.399526,0.793875,0.531546


In [19]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_16/embeddings_max_nmf_16_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_max_16/embeddings_max_nmf_16_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 16)
(25000, 16)
val f1: 0.11314
test f1: 0.10988
Validation pairs with F1 > 0.5: 4
Those also with test F1 > 0.5: 4
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
1,GO:0003924,3,0.6,0.605398,0.859489,0.710407,0.623167,0.811069,0.704809
2,GO:0005525,3,0.6,0.665810,0.688830,0.677124,0.686217,0.667618,0.676790
0,abc transporter,12,0.8,1.000000,0.571429,0.727273,1.000000,0.469136,0.638655
3,protein kinase,14,0.6,0.443511,0.575248,0.500862,0.475758,0.573516,0.520083


In [20]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_32/embeddings_max_nmf_32_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_max_32/embeddings_max_nmf_32_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 32)
(25000, 32)
val f1: 0.13274
test f1: 0.12799
Validation pairs with F1 > 0.5: 6
Those also with test F1 > 0.5: 6
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,abc transporter,29,0.60,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,GO:0003924,16,0.50,0.909627,0.844891,0.876064,0.907368,0.822519,0.862863
2,GO:0005525,16,0.50,0.986248,0.667553,0.796193,0.987368,0.669044,0.797619
3,GO:0003925,16,0.60,0.437632,0.945205,0.598266,0.394366,0.923077,0.552632
4,transmembrane,11,0.15,0.436639,0.707832,0.540105,0.441726,0.708776,0.544258
5,GO:0016887,31,0.60,0.750733,0.387879,0.511489,0.776119,0.394537,0.523139


In [21]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_64/embeddings_max_nmf_64_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_max_64/embeddings_max_nmf_64_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 64)
(25000, 64)
val f1: 0.16258
test f1: 0.15459
Validation pairs with F1 > 0.5: 16
Those also with test F1 > 0.5: 15
Survival rate: 0.9375


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,abc transporter,47,0.60,1.000000,0.982143,0.990991,1.000000,0.975309,0.987500
1,GO:0003924,39,0.50,0.907115,0.837591,0.870968,0.910870,0.799618,0.851626
2,GO:0005525,39,0.50,0.984190,0.662234,0.791733,0.993478,0.651926,0.787252
7,nudix hydrolase,34,0.80,0.884615,0.418182,0.567901,0.966667,0.568627,0.716049
3,protein kinase,63,0.60,0.723204,0.607921,0.660570,0.739374,0.603653,0.664656
4,GO:0106310,63,0.60,0.639576,0.675373,0.656987,0.633110,0.666667,0.649455
6,GO:0004674,63,0.60,0.559482,0.637584,0.595985,0.571588,0.637157,0.602594
5,GO:0003925,39,0.60,0.478365,0.908676,0.626772,0.421875,0.890110,0.572438
11,Zinc finger,8,0.50,0.643154,0.488343,0.555158,0.657500,0.501271,0.568854
9,transmembrane,3,0.15,0.458610,0.734627,0.564695,0.455265,0.729174,0.560548


In [22]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_128/embeddings_max_nmf_128_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_max_128/embeddings_max_nmf_128_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 128)
(25000, 128)
val f1: 0.18153
test f1: 0.17196
Validation pairs with F1 > 0.5: 21
Those also with test F1 > 0.5: 19
Survival rate: 0.9047619047619048


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,abc transporter,82,0.6,0.949153,1.000000,0.973913,0.987179,0.950617,0.968553
1,helicase,77,0.6,0.985714,0.766667,0.862500,0.951724,0.811765,0.876190
2,protein kinase,55,0.6,0.850279,0.753465,0.798950,0.850886,0.745205,0.794547
6,nudix hydrolase,70,0.8,0.969697,0.581818,0.727273,1.000000,0.647059,0.785714
3,GO:0003924,103,0.5,0.899510,0.669708,0.767782,0.905128,0.673664,0.772429
5,GO:0106310,53,0.6,0.673846,0.817164,0.738617,0.684411,0.848057,0.757496
7,GO:0005525,103,0.5,0.990196,0.537234,0.696552,0.994872,0.553495,0.711274
4,ig-like,40,0.6,0.723831,0.766509,0.744559,0.709756,0.708029,0.708892
8,GO:0004674,53,0.6,0.604103,0.790604,0.684884,0.607414,0.796758,0.689320
9,Zinc finger,78,0.5,0.745000,0.563327,0.641550,0.766724,0.567980,0.652555


In [35]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_320/embeddings_max_nmf_320_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_max_320/embeddings_max_nmf_320_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 320)
(25000, 320)
val f1: 0.14656
test f1: 0.1319
Validation pairs with F1 > 0.5: 5
Those also with test F1 > 0.5: 4
Survival rate: 0.8


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,279,0.6,0.563107,0.861386,0.681018,0.557143,0.854795,0.674595
1,GO:0106310,61,0.6,0.514907,0.794776,0.624939,0.522710,0.799764,0.632216
2,GO:0004674,61,0.6,0.465753,0.775839,0.582075,0.474211,0.768080,0.586387
3,2.7,279,0.6,0.552751,0.523605,0.537783,0.543452,0.535484,0.539439


Mean

In [23]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_8/embeddings_mean_nmf_8_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_8/embeddings_mean_nmf_8_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 8)
(25000, 8)
val f1: 0.07821
test f1: 0.07539
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,transmembrane,3,0.15,0.412681,0.909997,0.567846,0.413322,0.907268,0.567918
1,GO:0005634,7,0.50,0.491562,0.653255,0.560990,0.490535,0.648090,0.558412


In [24]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_16/embeddings_mean_nmf_16_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_16/embeddings_mean_nmf_16_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 16)
(25000, 16)
val f1: 0.08925
test f1: 0.08579
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,transmembrane,6,0.15,0.622123,0.812436,0.704655,0.616899,0.810936,0.700733
1,GO:0005634,11,0.15,0.454555,0.601604,0.517843,0.461102,0.607449,0.524254


In [25]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_32/embeddings_mean_nmf_32_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_32/embeddings_mean_nmf_32_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 32)
(25000, 32)
val f1: 0.11962
test f1: 0.11278
Validation pairs with F1 > 0.5: 4
Those also with test F1 > 0.5: 4
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,transmembrane,4,0.15,0.726007,0.671762,0.697832,0.720153,0.676208,0.697489
1,GO:0003925,28,0.50,0.573209,0.840183,0.681481,0.546468,0.807692,0.651885
3,GO:0005634,11,0.15,0.432610,0.698926,0.534428,0.430012,0.695664,0.531492
2,GO:0003924,28,0.50,0.750779,0.439781,0.554661,0.776952,0.398855,0.527112


In [26]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_64/embeddings_mean_nmf_64_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_64/embeddings_mean_nmf_64_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 64)
(25000, 64)
val f1: 0.14824
test f1: 0.14184
Validation pairs with F1 > 0.5: 8
Those also with test F1 > 0.5: 6
Survival rate: 0.75


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,pyridoxal 5'-phosphate,59,0.60,0.913907,0.605263,0.728232,0.945946,0.590717,0.727273
1,transmembrane,4,0.15,0.917037,0.531604,0.673046,0.902453,0.529654,0.667531
3,GO:0005506,22,0.80,0.903226,0.408759,0.562814,0.897143,0.422043,0.574040
4,heme,22,0.80,0.887097,0.392857,0.544554,0.862857,0.393229,0.540250
5,GO:0005634,12,0.15,0.450916,0.672421,0.539830,0.450474,0.665761,0.537356
7,GO:0007186,37,0.50,0.546448,0.478469,0.510204,0.553719,0.476303,0.512102


In [27]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_128/embeddings_mean_nmf_128_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_128/embeddings_mean_nmf_128_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 128)
(25000, 128)
val f1: 0.16798
test f1: 0.15855
Validation pairs with F1 > 0.5: 15
Those also with test F1 > 0.5: 11
Survival rate: 0.7333333333333333


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,transmembrane,4,0.15,0.910797,0.575232,0.705127,0.907647,0.577820,0.706116
1,pyridoxal 5'-phosphate,73,0.60,0.715686,0.640351,0.675926,0.719212,0.616034,0.663636
3,GO:0005506,77,0.80,1.000000,0.459854,0.630000,1.000000,0.456989,0.627306
2,GO:0003925,50,0.50,0.560000,0.831050,0.669118,0.498258,0.785714,0.609808
4,heme,77,0.80,0.989418,0.445238,0.614122,0.964706,0.427083,0.592058
6,fad,58,0.50,0.673993,0.518310,0.585987,0.670251,0.523810,0.588050
5,GO:0004252,101,0.50,0.794595,0.498305,0.612500,0.804598,0.457516,0.583333
7,GO:0005634,17,0.15,0.455212,0.759141,0.569143,0.458575,0.750850,0.569396
10,ig-like,87,0.60,0.645485,0.455189,0.533887,0.666667,0.452555,0.539130
11,1.14,77,0.80,0.835979,0.390123,0.531987,0.847059,0.381963,0.526508


In [36]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_320/embeddings_mean_nmf_320_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_320/embeddings_mean_nmf_320_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 320)
(25000, 320)
val f1: 0.18239
test f1: 0.16985
Validation pairs with F1 > 0.5: 23
Those also with test F1 > 0.5: 20
Survival rate: 0.8695652173913043


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,nudix hydrolase,273,0.50,1.000000,0.709091,0.829787,1.000000,0.686275,0.813953
3,krab,147,0.60,0.650000,0.750000,0.696429,0.788462,0.672131,0.725664
2,GO:0003924,148,0.50,0.877005,0.598540,0.711497,0.868805,0.568702,0.687428
7,fad,249,0.50,0.940541,0.490141,0.644444,0.940299,0.529412,0.677419
1,GO:0003925,148,0.60,0.645695,0.890411,0.748560,0.562044,0.846154,0.675439
5,GO:0005506,174,0.50,1.000000,0.508516,0.674194,0.994652,0.500000,0.665474
4,ig-like,99,0.50,0.708543,0.665094,0.686131,0.713483,0.618005,0.662321
10,pyridoxal 5'-phosphate,176,0.60,0.690355,0.596491,0.640000,0.715026,0.582278,0.641860
6,heme,174,0.50,0.985646,0.490476,0.655008,0.962567,0.468750,0.630473
8,GO:0005525,148,0.50,0.967914,0.481383,0.642984,0.959184,0.469330,0.630268


Min

In [28]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_8/embeddings_min_nmf_8_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_min_8/embeddings_min_nmf_8_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 8)
(25000, 8)
val f1: 0.08049
test f1: 0.07697
Validation pairs with F1 > 0.5: 0
Those also with test F1 > 0.5: 0
Survival rate: 0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


In [29]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_16/embeddings_min_nmf_16_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_min_16/embeddings_min_nmf_16_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 16)
(25000, 16)
val f1: 0.09492
test f1: 0.09091
Validation pairs with F1 > 0.5: 4
Those also with test F1 > 0.5: 4
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003924,7,0.60,0.538373,0.857664,0.661506,0.544767,0.824427,0.656036
1,GO:0005525,7,0.60,0.593356,0.688830,0.637538,0.601513,0.680456,0.638554
2,GO:0005634,1,0.50,0.465871,0.626206,0.534269,0.469829,0.628653,0.537759
3,transmembrane,3,0.15,0.396461,0.765888,0.522468,0.393812,0.767912,0.520628


In [30]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_32/embeddings_min_nmf_32_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_min_32/embeddings_min_nmf_32_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 32)
(25000, 32)
val f1: 0.12378
test f1: 0.11803
Validation pairs with F1 > 0.5: 5
Those also with test F1 > 0.5: 5
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003924,17,0.80,0.871391,0.605839,0.714747,0.870091,0.549618,0.673684
1,GO:0003925,17,0.80,0.517060,0.899543,0.656667,0.477341,0.868132,0.615984
2,GO:0005525,17,0.80,0.947507,0.480053,0.637246,0.945619,0.446505,0.606589
3,GO:0016887,22,0.60,0.676026,0.474242,0.557435,0.682222,0.465857,0.553652
4,transmembrane,5,0.15,0.458029,0.656991,0.539759,0.460607,0.658382,0.542017


In [31]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_64/embeddings_min_nmf_64_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_min_64/embeddings_min_nmf_64_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 64)
(25000, 64)
val f1: 0.15366
test f1: 0.14584
Validation pairs with F1 > 0.5: 12
Those also with test F1 > 0.5: 11
Survival rate: 0.9166666666666666


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,abc transporter,46,0.60,0.981481,0.946429,0.963636,0.974359,0.938272,0.955975
1,GO:0003924,11,0.50,0.896887,0.841241,0.868173,0.898947,0.814885,0.854855
3,helicase,26,0.60,0.822485,0.772222,0.796562,0.818750,0.770588,0.793939
2,GO:0005525,11,0.50,0.982490,0.671543,0.797788,0.981053,0.664765,0.792517
4,protein kinase,56,0.50,0.655488,0.851485,0.740741,0.677054,0.873059,0.762665
5,GO:0106310,56,0.50,0.514482,0.839552,0.637996,0.518414,0.862191,0.647501
7,GO:0004674,56,0.50,0.464177,0.817450,0.592124,0.475212,0.836658,0.606143
9,2.7,56,0.50,0.646341,0.519926,0.576283,0.654391,0.541935,0.592878
6,GO:0003925,11,0.60,0.460317,0.926941,0.615152,0.414322,0.890110,0.565445
10,transmembrane,24,0.15,0.484765,0.666781,0.561388,0.477599,0.661467,0.554693


In [32]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_128/embeddings_min_nmf_128_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_min_128/embeddings_min_nmf_128_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 128)
(25000, 128)
val f1: 0.16363
test f1: 0.15533
Validation pairs with F1 > 0.5: 12
Those also with test F1 > 0.5: 11
Survival rate: 0.9166666666666666


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,abc transporter,42,0.50,1.000000,0.946429,0.972477,0.975309,0.975309,0.975309
1,helicase,45,0.60,0.882716,0.794444,0.836257,0.884615,0.811765,0.846626
2,GO:0003924,54,0.50,0.907127,0.766423,0.830861,0.910194,0.715649,0.801282
3,GO:0005525,54,0.50,0.989201,0.609043,0.753909,0.995146,0.584879,0.736748
5,krab,84,0.80,0.630769,0.788462,0.700855,0.714286,0.737705,0.725806
4,protein kinase,57,0.60,0.680144,0.749505,0.713142,0.697714,0.752511,0.724077
6,GO:0106310,57,0.60,0.566038,0.783582,0.657277,0.580864,0.808009,0.675862
8,GO:0004674,57,0.60,0.504043,0.753020,0.603875,0.520745,0.766833,0.620272
7,GO:0003925,54,0.60,0.505263,0.876712,0.641068,0.457227,0.851648,0.595010
10,2.7,57,0.60,0.660377,0.450644,0.535714,0.674005,0.466862,0.551629


In [37]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_320/embeddings_min_nmf_320_val_features.npz",
    allow_pickle=True,)

test_data = np.load(
    "Extraction/test_features/embeddings_min_320/embeddings_min_nmf_320_test_features.npz",
    allow_pickle=True,)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
maxs = X_val.max(axis=0)
A_val = X_val / (maxs + 1e-8)
A_test = X_test / (maxs + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs

(25000, 320)
(25000, 320)
val f1: 0.12986
test f1: 0.117
Validation pairs with F1 > 0.5: 4
Those also with test F1 > 0.5: 3
Survival rate: 0.75


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003924,30,0.6,0.805353,0.604015,0.690302,0.806878,0.582061,0.676275
1,GO:0005525,30,0.6,0.873479,0.477394,0.617369,0.870370,0.469330,0.609824
2,GO:0003925,30,0.6,0.454988,0.853881,0.593651,0.394180,0.818681,0.532143
